In [1]:
# main.py
# ─────────────────────────────────────────────────────────────────────────────
# Orchestrates the full solenoid design-space study.
#
#   python main.py            # run everything
# ─────────────────────────────────────────────────────────────────────────────


import importlib
import numpy as np

import solenoid_lib
importlib.reload(solenoid_lib)

import config
importlib.reload(config)

import grid
importlib.reload(grid)

import io_grids
importlib.reload(io_grids)

import plots_thickness
importlib.reload(plots_thickness)

import plots_area
importlib.reload(plots_area)

#def main():
    #print("=== Thickness scan (Ri vs Th) ===")
    #th_grids = grid.build_thickness_grid()
    #io_grids.save_grids_long(th_grids, "thickness_grids.csv")
    #plots_thickness.plot_all(th_grids)

#    print("\n=== Area scan (Ri vs A) ===")
#    a_grids = grid.build_area_grid()
#    io_grids.save_grids_long(a_grids, "area_grids.csv")
#    plots_area.plot_all(a_grids)
#    
#    return a_grids


#a_grids = main()

LENGTHS_M = [1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
#LENGTHS_M = [1.0]

def main():
    all_grids = {}
    for L in LENGTHS_M:
        #print(f"\n=== Area scan  |  solenoid_length = {L*1e3:.0f} mm ===")
        solenoid_lib.solenoid_length = L
        label = f"L{int(L*1e3)}mm"
        a_grids = grid.build_area_grid()
        io_grids.save_grids_long(a_grids, f"area_grids_{label}.csv", solenoid_length=L)
        plots_area.plot_all(a_grids, label=label, solenoid_length_m=L)
        all_grids[L] = a_grids
    return all_grids

all_grids = main()

# print(f"Scan log10 max thickness scan: {np.nanmax(np.log10(th_grids['scan'])):.2f}")
#print(f"Scan log10 max area scan:      {np.nanmax(np.log10(a_grids['scan'])):.2f}")
#print(f"Th max in area scan:      {np.nanmax(a_grids['th'])*1e3:.1f} mm")
#print(f"Th max in thickness scan: {np.nanmax(th_grids['th'])*1e3:.1f} mm")
#print(f"Ri max in area scan:      {np.nanmax(a_grids['ri'])*1e3:.1f} mm")

#print(f"Th at RI_MAX, A_MAX = {(np.sqrt(1.0**2 + config.A_MAX/np.pi) - 1.0)*1e3:.1f} mm")



KeyboardInterrupt: 

In [15]:
import importlib
import solenoid_lib


importlib.reload(solenoid_lib)


from solenoid_lib import scan_time, dB_to_eta

s_per_year = 365.25 * 24 * 3600

print(f"{'Scenario':<40} {'Expected':>10}  {'DFSZ':>10} {'Ratio':>10}")
print("-" * 74)

scenarios = [
    (16, 10, 20e6, -20, 6.2,  'Baseline'),
    (29, 10, 20e6, -5,  3.2,  'Stronger magnet + higher noise'),
    (16,  8, 20e6, -25, 7.3,  'Lower noise + lower volume'),
    (16, 17,  2e6, -20, 10.6, 'Higher volume + lower Q'),
    (26, 10,  2e6, -20, 8.9,  'Stronger magnet + lower Q'),
]

for (B0, V_m3, Q, eta_dB, expected, name) in scenarios:
    eta_A  = dB_to_eta(eta_dB)
    t_dfsz = scan_time(B0, V_m3, Q=Q, eta_A=eta_A, model='DFSZ')
    ratio  = expected / t_dfsz
    print(f"{name:<40} {expected:>10.1f} {t_dfsz:>10.2f} {ratio:>10.2f}")

Scenario                                   Expected        DFSZ      Ratio
--------------------------------------------------------------------------
Baseline                                        6.2       4.12       1.51
Stronger magnet + higher noise                  3.2       2.15       1.49
Lower noise + lower volume                      7.3       4.87       1.50
Higher volume + lower Q                        10.6       7.02       1.51
Stronger magnet + lower Q                       8.9       5.90       1.51
